In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import joblib

In [ ]:
dpath = "D:/UKB/four/"
outfile = os.path.join(dpath, "XGB_feature_importance.csv")
model_path = os.path.join(dpath, "best_xgb_model.pkl")  # 模型保存路径
df = pd.read_csv(r"D:\Rdata and workplace\整合.csv")
england_indices = df[df['Region'] == 'England'].index
X_england = df.loc[england_indices].drop(columns=['Participant.ID', 'status', 'Region'])
y_england = df.loc[england_indices, 'status']  # 
X_train = X_england
y_train = y_england
df_group = y_train
df_feature = X_train

# 筛选来自苏格兰和威尔士的参与者
external_indices = df[df['Region'].isin(['Scotland', 'Wales'])].index
# 获取苏格兰和威尔士参与者的数据
X_external = df.loc[external_indices].drop(columns=['Participant.ID', 'status', 'Region'])
y_external = df.loc[external_indices, 'status']  # 标签
print("外部验证集形状:", X_external.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from joblib import dump
from sklearn.metrics import roc_auc_score  # 添加这一行
# 初始化参数
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
   'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

# 存储数据容器
best_model_info = {
    'model': None,
    'fold': 0,
    'auc': 0,
    'val_idx': None,  # 存储验证集索引
    'fpr': None,
    'tpr': None
}
cv_metrics = []
all_fpr = []
all_tpr = []

# ================= 交叉验证流程 =================
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold+1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 模型训练
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    
    # 计算指标
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    # 存储结果
    cv_metrics.append(fold_auc)
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    all_fpr.append(fpr)
    all_tpr.append(tpr)
    
    # 更新最佳模型
    if fold_auc > best_model_info['auc']:
        best_model_info.update({
            'model': model,
            'fold': fold+1,
            'auc': fold_auc,
            'val_idx': val_idx,  # 保存验证集索引
            'fpr': fpr,
            'tpr': tpr
        })
        print(f"New best model at Fold {fold+1}, AUC = {fold_auc:.4f}")

# 保存最佳模型
dump(best_model_info['model'], 'best_model.joblib')
print(f"\nBest model from Fold {best_model_info['fold']}, AUC = {best_model_info['auc']:.4f}")

# ================= ROC曲线可视化 =================
plt.figure(figsize=(6, 6))

# 绘制各折ROC曲线
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i in range(5):
    plt.plot(all_fpr[i], all_tpr[i], 
             color=colors[i], lw=1, alpha=0.3,
             label=f'Fold {i+1} (AUC = {cv_metrics[i]:.2f})')

# 计算平均ROC曲线
mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(5):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= 5
mean_auc = auc(mean_fpr, mean_tpr)

# 绘制平均曲线
plt.plot(mean_fpr, mean_tpr, color='b', lw=2,
         label=f'Mean AUC = {mean_auc:.2f} ')


# 绘制外部验证曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='g', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves', fontsize=14)
plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.grid(alpha=0.3)
plt.tight_layout()

# 保存为PDF（关键修改：在plt.show()之前保存）
plt.savefig('roc_curve.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 混淆矩阵可视化 =================
# 获取最佳折数据
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_best = y_train.iloc[best_model_info['val_idx']]

# 生成预测结果
y_val_pred = best_model_info['model'].predict(X_val_best)
y_ext_pred = best_model_info['model'].predict(X_external)

# 创建画布
plt.figure(figsize=(12, 5))

# 内部验证矩阵
plt.subplot(1, 2, 1)
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(cm_val, display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title(f'Best Fold ({best_model_info["fold"]})\nAUC = {best_model_info["auc"]:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_val[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_val[i, j] > cm_val.max()/2 else "black")

# 外部验证矩阵
plt.subplot(1, 2, 2)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', ax=plt.gca(), colorbar=False)
plt.title(f'External Validation\nAUC = {roc_auc_ext:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_ext[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_ext[i, j] > cm_ext.max()/2 else "black")

plt.tight_layout()

# 保存为PDF（关键修改：在plt.show()之前保存）
plt.savefig('整合confusion_matrix.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 性能报告 =================
print("\n" + "="*55)
print(f"{' Internal Validation Report ':=^55}")
print(classification_report(y_val_best, y_val_pred, target_names=['Healthy', 'Stroke']))

print("\n" + "="*55)
print(f"{' External Validation Report ':=^55}")
print(classification_report(y_external, y_ext_pred, target_names=['Healthy', 'Stroke']))
# 获取所有样本的索引和标签
all_indices = df.index
all_labels = df['status']
all_regions = df['Region']
all_participant_ids = df['Participant.ID']
all_features = df.drop(columns=['Participant.ID', 'status', 'Region'])

# 使用最佳模型进行预测概率（打分）
all_probas = best_model_info['model'].predict_proba(all_features)[:, 1]

# 创建结果数据框
results = pd.DataFrame({
    'Participant.ID': all_participant_ids,
    'Region': all_regions,
    'Status': all_labels,
    'Predicted Probability': all_probas
})

# 按区域分组排序（可选）
results = results.sort_values(by='Region')

# 保存为CSV文件
results.to_csv(os.path.join(dpath, "XGB_predictions.csv"), index=False)
print(f"每个样本的预测概率已保存到 {os.path.join(dpath, 'XGB_predictions.csv')}")

In [ ]:
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}
# 存储最佳模型信息
best_model = None
best_auc = 0
best_fold = 0
cv_metrics = []


for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold+1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 训练模型
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, 
              eval_set=[(X_val, y_val)],
              verbose=0)
    
    # 验证集预测
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    cv_metrics.append(fold_auc)
    print(f"Fold {fold+1} AUC: {fold_auc:.4f}")
    
    # 更新最佳模型
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold + 1
        print(f"New best model found at Fold {best_fold} with AUC: {best_auc:.4f}")

# 保存最佳模型
joblib.dump(best_model, model_path)
print(f"\nBest model saved from Fold {best_fold} with AUC: {best_auc:.4f}")

# 交叉验证结果
print(f"\n=== Cross-validation Results ===")
print(f"Mean AUC: {np.mean(cv_metrics):.4f} (±{np.std(cv_metrics):.4f})")

# 使用最佳模型进行外部验证
print("\n=== External Validation with Best Model ===")
y_ext_proba = best_model.predict_proba(X_external)[:, 1]
ext_auc = roc_auc_score(y_external, y_ext_proba)
print(f"External Validation AUC: {ext_auc:.4f}")

# 1. 准备最佳折的内部验证集数据
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_count = 0

for train_idx, val_idx in cv.split(X_train, y_train):
    fold_count += 1
    if fold_count == best_fold:
        X_val_best = X_train.iloc[val_idx]
        y_val_best = y_train.iloc[val_idx]
        break

# 2. 预测概率
y_val_proba = best_model.predict_proba(X_val_best)[:, 1]
y_ext_proba = best_model.predict_proba(X_external)[:, 1]

# 3. 创建画布
plt.figure(figsize=(14, 6))

# ========== ROC曲线对比 ==========
plt.subplot(1, 2, 1)

# 内部验证集ROC
fpr_val, tpr_val, _ = roc_curve(y_val_best, y_val_proba)
roc_auc_val = auc(fpr_val, tpr_val)
plt.plot(fpr_val, tpr_val, color='blue', 
         label=f'Internal (Fold {best_fold})\nAUC = {roc_auc_val:.2f}')

# 外部验证集ROC
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='orange',
         label=f'External\nAUC = {roc_auc_ext:.2f}')

# 参考线
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')

# 1. 准备最佳折的内部验证集数据
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    if fold + 1 == best_fold:  # 找到最佳折
        X_val_best = X_train.iloc[val_idx]
        y_val_best = y_train.iloc[val_idx]
        break

# 2. 预测结果
y_val_pred = best_model.predict(X_val_best)
y_ext_pred = best_model.predict(X_external)

# 3. 创建混淆矩阵对比可视化
plt.figure(figsize=(14, 6))

# ========== 内部验证集混淆矩阵 ==========
plt.subplot(1, 2, 1)
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val,
                               display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', values_format='d', ax=plt.gca(), colorbar=False)
plt.title(f'Internal Validation (Best Fold {best_fold})\nAUC = {best_auc:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_val[i, j]:d}",
                ha="center", va="center",
                color="white" if cm_val[i, j] > cm_val.max()/2 else "black")

# ========== 外部验证集混淆矩阵 ==========
plt.subplot(1, 2, 2)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(confusion_matrix=cm_ext,
                               display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', values_format='d', ax=plt.gca(), colorbar=False)
plt.title(f'External Validation\nAUC = {ext_auc:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_ext[i, j]:d}",
                ha="center", va="center",
                color="white" if cm_ext[i, j] > cm_ext.max()/2 else "black")

plt.tight_layout()
plt.show()

# 4. 打印性能报告
print("\n" + "="*50)
print(f"{' INTERNAL VALIDATION (BEST FOLD) ':=^50}")
print("="*50)
print(f"AUC: {best_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val_best, y_val_pred, 
                          target_names=['Healthy', 'Stroke']))

print("\n" + "="*50)
print(f"{' EXTERNAL VALIDATION ':=^50}")
print("="*50)
print(f"AUC: {ext_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_external, y_ext_pred,
                          target_names=['Healthy', 'Stroke']))

In [ ]:
# 计算特征的Spearman相关性矩阵
correlation_matrix = df_feature.corr(method='spearman')

# 打印相关性矩阵
print(correlation_matrix)

# 保存相关性矩阵为CSV文件
correlation_matrix.to_csv(dpath + 'Stroke_hc_correlation_matrix.csv', index=True)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist, squareform

pdf_output_file = r'D:\UKB\four\Stroke_hc_correlation_matrix_heatmap.pdf'

# 将相关性矩阵转换为距离矩阵 (1 - 绝对值相关性)(相关性越强（|r|→1），距离越近（→0）)
distance_matrix = 1 - np.abs(correlation_matrix)

# 计算层次聚类的链式方法 (使用 Ward 方法)
# pdist 用于计算距离矩阵的压缩形式
dist_array = squareform(distance_matrix)  # 转换为压缩距离矩阵
dist_linkage = hierarchy.linkage(dist_array, method='ward')

In [ ]:
# 创建单个子图用于绘制树状图
fig, ax = plt.subplots(figsize=(30, 16))
# 计算层次聚类并生成树状图 (Dendrogram)
dendro = hierarchy.dendrogram(dist_linkage, labels=correlation_matrix.columns, ax=ax)
# 设置树状图的 x 轴标签
ax.set_xticklabels(dendro["ivl"], rotation=60, fontsize=4, horizontalalignment='right')
# 绘制水平线以标识聚类阈值 (可以根据需要调整y的值)
ax.axhline(y=0.5, color='r', linewidth=2, linestyle='--')
# 保存图像为文件 (PDF 或 PNG，或其他格式)
plt.savefig(pdf_output_file, dpi=300, format='pdf')  # 保存为PDF文件，确保图像的高分辨率
# 显示图像
plt.show()

In [ ]:
from collections import defaultdict
from scipy.cluster import hierarchy

# 聚类，距离阈值为0.5，按簇划分
cluster_ids = hierarchy.fcluster(dist_linkage, 0.5, criterion="distance")

# 创建字典，将簇ID映射到特征的索引(每个特征被分配到一个簇ID)
cluster_id_to_feature_ids = defaultdict(list)

# 将特征的索引按簇ID分类(同一簇内的特征高度相关（Spearman |r| > 0.5）)
for idx, cluster_id in enumerate(cluster_ids):
    cluster_id_to_feature_ids[cluster_id].append(idx)

# 输出聚类结果（每个簇的特征索引）
for cluster_id, feature_ids in cluster_id_to_feature_ids.items():
    print(f"Cluster {cluster_id}: Feature indices {feature_ids}")

# 将特征的簇ID与特征名称对应
feature_names = df_feature.columns.tolist()  # 获取特征名称
cluster_data = pd.DataFrame({
    'Feature': feature_names,
    'ClusterID': cluster_ids
})

# 输出文件路径
output_csv = r'D:\UKB\four\Stroke_feature_clusters.csv'
# 将结果保存为CSV文件
cluster_data.to_csv(output_csv, index=False)
print(f"聚类结果已保存至: {output_csv}")

In [ ]:
dpath = "D:/UKB/four/"

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from tqdm import tqdm  # 添加进度条
import os

output_file = os.path.join(dpath, 'RNA_auc_scores.csv')

scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

# 3. 模型参数配置
params = {
   'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

# 4. 五折交叉验证设置
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 5. 单变量评估函数（优化版）
def calculate_auc_for_feature(feature, X, y, cv, params):
    """计算单个特征的交叉验证AUC"""
    auc_scores = []
    
    for train_idx, val_idx in cv.split(X, y):
        # 确保数据是二维的
        X_train = X.iloc[train_idx][[feature]].values.reshape(-1, 1)
        X_val = X.iloc[val_idx][[feature]].values.reshape(-1, 1)
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        # 模型训练和预测
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_proba))
    
    return np.mean(auc_scores)

# 6. 特征评估（带进度条）
auc_scores = {}
for feature in tqdm(df_feature.columns, desc="Evaluating features"):
    auc_scores[feature] = calculate_auc_for_feature(feature, df_feature, df_group, cv, params)

# 7. 保存结果
auc_df = pd.DataFrame(list(auc_scores.items()), columns=['Feature', 'AUC'])
auc_df = auc_df.sort_values('AUC', ascending=False)  # 按AUC降序排列
auc_df.to_csv(output_file, index=False)

print(f"\nAUC scores saved to: {output_file}")
print("\nTop 10 features by AUC:")
print(auc_df.head(10))

# 8. 可视化Top特征
plt.figure(figsize=(10, 6))
plt.barh(auc_df['Feature'].head(20)[::-1], auc_df['AUC'].head(20)[::-1])
plt.xlabel('AUC Score')
plt.title('Top 20 Predictive Features (5-fold CV)')
plt.tight_layout()
plt.show()

In [ ]:
dpath = "D:/UKB/four/"

In [ ]:
feature_auc = pd.read_csv(dpath + 'RNA_auc_scores.csv', encoding='GBK')
feature_clus = pd.read_csv(dpath + 'Stroke_feature_clusters.csv', encoding='GBK')

feature_sel = pd.merge(feature_auc, feature_clus, how='left', on="Feature")
best_features = feature_sel.loc[feature_sel.groupby('ClusterID')['AUC'].idxmax()]

# 输出选择后的特征
output_file = dpath + 'RNA_best_features.csv'
best_features.to_csv(output_file, index=False)
feature482 = best_features['Feature']
feature482

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix, 
                            roc_curve, precision_score, recall_score, f1_score, 
                            matthews_corrcoef, ConfusionMatrixDisplay)
import xgboost as xgb
import joblib
from tqdm import tqdm

# 1. 数据准备
feature482 = best_features['Feature']
X_train_selected = X_train[feature482]
X_external_selected = X_external[feature482]
X_train_selected

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                            roc_curve, auc, ConfusionMatrixDisplay)
import xgboost as xgb

# 数据准备
feature482 = best_features['Feature']
X_train_selected = X_train[feature482]
X_external_selected = X_external[feature482]

# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

# 模型参数
params = {
    'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

# 五折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_model = None
best_auc = 0
best_fold = 0

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_selected, y_train), 1):
    X_tr, X_val = X_train_selected.iloc[train_idx], X_train_selected.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold
        X_best_val = X_val
        y_best_val = y_val

# 内部验证可视化
plt.figure(figsize=(12, 5))

# 内部验证ROC
plt.subplot(1, 2, 1)
y_val_proba = best_model.predict_proba(X_best_val)[:, 1]
fpr, tpr, _ = roc_curve(y_best_val, y_val_proba)
plt.plot(fpr, tpr, label=f'Fold {best_fold} (AUC={best_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Internal Validation ROC')
plt.legend()

# 内部验证混淆矩阵
plt.subplot(1, 2, 2)
y_val_pred = best_model.predict(X_best_val)
cm = confusion_matrix(y_best_val, y_val_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Stroke', 'Stroke'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Internal Validation CM')

plt.tight_layout()
plt.show()

# 外部验证可视化
plt.figure(figsize=(12, 5))

# 外部验证ROC
plt.subplot(1, 2, 1)
y_ext_proba = best_model.predict_proba(X_external_selected)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 'r', label=f'External (AUC={roc_auc_ext:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('External Validation ROC')
plt.legend()

# 外部验证混淆矩阵
plt.subplot(1, 2, 2)
y_ext_pred = best_model.predict(X_external_selected)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['No Stroke', 'Stroke'])
disp_ext.plot(cmap='Reds', values_format='d')
plt.title('External Validation CM')

plt.tight_layout()
plt.show()

In [ ]:
df_feature = df_feature[feature482]

df_feature.to_csv('D:/UKB/four/rmRmMultiColin_feature482.csv', index=False)

df_feature

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from collections import Counter
# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
# 设置模型参数
params = {
    'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}
# 交叉验证设置
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 函数：标准化重要性
def normal_imp(mydict):
    mysum = sum(mydict.values())
    for key in mydict.keys():
        mydict[key] = mydict[key] / mysum
    return mydict

# 初始化重要性计数器
tg_imp_cv = Counter()
shap_imp_cv = np.zeros(df_feature.shape[1])  # 初始化SHAP重要性数组

# 交叉验证过程
for train_idx, test_idx in cv.split(df_feature, df_group):
    X_train, X_test = df_feature.iloc[train_idx, :], df_feature.iloc[test_idx, :]
    y_train, y_test = df_group.iloc[train_idx], df_group.iloc[test_idx]

    # 训练 XGB 分类器
    my_xgb = XGBClassifier(**params)
    my_xgb.fit(X_train, y_train)

    # 计算总增益重要性
    totalgain_imp = my_xgb.feature_importances_  # 直接使用 feature_importances_
    totalgain_imp = dict(zip(df_feature.columns, totalgain_imp.tolist()))

    # # 计算总覆盖率重要性，xgb没有办法计算
    # totalcover_imp = my_xgb.booster_.feature_importance(importance_type='split')
    # totalcover_imp = dict(zip(df_feature.columns, totalcover_imp.tolist()))

    # 更新重要性计数器
    tg_imp_cv += Counter(normal_imp(totalgain_imp))

    # 计算 SHAP 值
    explainer = shap.TreeExplainer(my_xgb)
    shap_values = explainer.shap_values(X_test)
    # 取绝对值的平均 SHAP 值，确保分母合理
    shap_values_mean = np.mean(np.abs(shap_values), axis=0)
    shap_imp_cv += shap_values_mean / np.sum(shap_values_mean)  # 归一化

In [ ]:
shap_values_mean.shape

In [ ]:
# 创建 SHAP 重要性数据框
shap_imp_df = pd.DataFrame({
    'Analytes': df_feature.columns,
    'ShapValues_cv': shap_imp_cv / 10
})
shap_imp_df.sort_values(by='ShapValues_cv', ascending=False, inplace=True)

# 计算基本统计信息
stats_summary = {
    'Mean': shap_imp_df['ShapValues_cv'].mean(),
    'Std': shap_imp_df['ShapValues_cv'].std(),
    'Min': shap_imp_df['ShapValues_cv'].min(),
    'Max': shap_imp_df['ShapValues_cv'].max(),
    '25%': shap_imp_df['ShapValues_cv'].quantile(0.25),
    '50% (Median)': shap_imp_df['ShapValues_cv'].median(),
    '75%': shap_imp_df['ShapValues_cv'].quantile(0.75)
}

# 打印统计信息
print("SHAP Values CV Statistics:")
for stat, value in stats_summary.items():
    print(f"{stat}: {value:.4f}")

In [ ]:
# 创建总增益重要性数据框
tg_imp_cv = normal_imp(tg_imp_cv)
tg_imp_df = pd.DataFrame({
    'Analytes': list(tg_imp_cv.keys()),
    'TotalGain_cv': list(tg_imp_cv.values())
})
tg_imp_df

In [ ]:
# 合并所有重要性数据框
my_imp_df = pd.merge(left=shap_imp_df, right=tg_imp_df, how='left', on=['Analytes'])

# 计算综合重要性
my_imp_df['Ensemble_cv'] = (my_imp_df['ShapValues_cv'] + my_imp_df['TotalGain_cv']) / 2
my_imp_df.sort_values(by='TotalGain_cv', ascending=False, inplace=True)

# 保存结果
my_imp_df.to_csv(outfile, index=False)

print('finished')

In [ ]:
# 筛选前90%增益特征，不只使用TotalGain_cv，使用Ensemble_cv
def get_imp_analy(my_imp_df, top_prop=0.5):
    imp_score, iter = 0, 0
    # 遍历 my_imp_df 的 Ensemble_cv 列，累加直到累计超过 top_prop
    while imp_score < top_prop and iter < len(my_imp_df):
        imp_score += my_imp_df.Ensemble_cv.iloc[iter]  # 使用 iloc 获取第 iter 行的值
        iter += 1
    return iter  # 返回累积达到 top_prop 时的行索引

# 调用函数，使用 Ensemble_cv
top_feature_count = get_imp_analy(my_imp_df, top_prop=0.9)

print(f"Number of proteins contributing to over 90% of overall information gains: {top_feature_count}")

# 获取前 top_feature_count 个特征
top_features = my_imp_df.iloc[:top_feature_count]

# 保存到 CSV 文件
dpath = 'D:/UKB/four/'
top_features.to_csv(dpath + 'Top_40_InfoGain_features.csv', index=False)

print('Top-ranked proteins saved to CSV.')

In [ ]:
# 获取前 top_feature_count 个特征
top_features = my_imp_df.iloc[:top_feature_count]

# 保存到 CSV 文件

top_features.to_csv(dpath + 'Top_40_InfoGain_features.csv', index=False)

print('Top-ranked proteins saved to CSV.')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import shap
from collections import Counter

dpath = 'D:/UKB/four/'
# 加载数据
top_features_df = pd.read_csv(dpath + 'Top_40_InfoGain_features.csv')
top_features = top_features_df['Analytes'].tolist()
top_features_df

In [ ]:
df = pd.read_csv(r"D:\Rdata and workplace\整合.csv")
england_indices = df[df['Region'] == 'England'].index
X_england = df.loc[england_indices].drop(columns=['Participant.ID', 'status', 'Region'])
y_england = df.loc[england_indices, 'status']  # 
X_train = X_england
y_train = y_england
df_group = y_train
df_feature = X_train

# 筛选来自苏格兰和威尔士的参与者
external_indices = df[df['Region'].isin(['Scotland', 'Wales'])].index
# 获取苏格兰和威尔士参与者的数据
X_external = df.loc[external_indices].drop(columns=['Participant.ID', 'status', 'Region'])
y_external = df.loc[external_indices, 'status']  # 标签
print("外部验证集形状:", X_external.shape)

In [ ]:
df_feature = X_train[top_features]
df_feature

In [ ]:
X_train = df_feature
X_train

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb

# 数据准备
X_train_selected = X_train[top_features]
X_external_selected = X_external[top_features]

# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

# 模型参数
params = {
    'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

# 五折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_model = None
best_auc = 0
best_fold = 0

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_selected, y_train), 1):
    X_tr, X_val = X_train_selected.iloc[train_idx], X_train_selected.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr)
    
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)
    
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model
        best_fold = fold
        X_best_val = X_val
        y_best_val = y_val

# 内部验证可视化
plt.figure(figsize=(12, 5))

# 内部ROC曲线
plt.subplot(1, 2, 1)
y_val_proba = best_model.predict_proba(X_best_val)[:, 1]
fpr, tpr, _ = roc_curve(y_best_val, y_val_proba)
plt.plot(fpr, tpr, label=f'Fold {best_fold} (AUC={best_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Internal Validation ROC')
plt.legend()

# 内部混淆矩阵
plt.subplot(1, 2, 2)
y_val_pred = best_model.predict(X_best_val)
cm = confusion_matrix(y_best_val, y_val_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Stroke', 'Stroke'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Internal Validation CM')

plt.tight_layout()
plt.show()

# 外部验证可视化
plt.figure(figsize=(12, 5))

# 外部ROC曲线
plt.subplot(1, 2, 1)
y_ext_proba = best_model.predict_proba(X_external_selected)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, 'r', label=f'External (AUC={roc_auc_ext:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('External Validation ROC')
plt.legend()

# 外部混淆矩阵
plt.subplot(1, 2, 2)
y_ext_pred = best_model.predict(X_external_selected)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['No Stroke', 'Stroke'])
disp_ext.plot(cmap='Reds', values_format='d')
plt.title('External Validation CM')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import scipy.stats
from scipy.stats import norm
from scipy import stats


# AUC comparison adapted from
# https://github.com/Netflix/vmaf/
def compute_midrank(x):
    """Computes midranks."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def fastDeLong(predictions_sorted_transposed, label_1_count):
    """
    The fast version of DeLong's method for computing the covariance of
    unadjusted AUC.
    Args:
       predictions_sorted_transposed: a 2D numpy.array[n_classifiers, n_examples]
          sorted such as the examples with label "1" are first
    Returns:
       (AUC value, DeLong covariance)
    Reference:
     @article{sun2014fast,
       title={Fast Implementation of DeLong's Algorithm for
              Comparing the Areas Under Correlated Receiver Operating Characteristic Curves},
       author={Xu Sun and Weichao Xu},
       journal={IEEE Signal Processing Letters},
       volume={21},
       number={11},
       pages={1389--1393},
       year={2014},
       publisher={IEEE}
     }
    """
    # Short variables are named as they are in the paper
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def calc_pvalue(aucs, covar):
    """Computes log(10) of p-values.
    Args:
       aucs: 1D array of AUCs
       covar: AUC DeLong covariances
    Returns:
       log10(pvalue)
    """
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, covar), l.T))  # 将 sigma 改为 covar
    return np.log10(2) + scipy.stats.norm.logsf(z, loc=0, scale=1) / np.log(10)


def compute_ground_truth_statistics(ground_truth):
    assert np.array_equal(np.unique(ground_truth), [0, 1])
    order = (-ground_truth).argsort()
    label_1_count = int(ground_truth.sum())
    return order, label_1_count


def delong_roc_variance(ground_truth, predictions):
    """
    Computes ROC AUC variance for a single set of predictions
    Args:
       ground_truth: np.array of 0 and 1
       predictions: np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = predictions[np.newaxis, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    assert len(aucs) == 1, "There is a bug in the code, please forward this to the developers"
    return aucs[0], delongcov


def delong_roc_test(ground_truth, predictions_one, predictions_two):
    """
    Computes log(p-value) for hypothesis that two ROC AUCs are different
    Args:
       ground_truth: np.array of 0 and 1
       predictions_one: predictions of the first model,
          np.array of floats of the probability of being class 1
       predictions_two: predictions of the second model,
          np.array of floats of the probability of being class 1
    """
    order, label_1_count = compute_ground_truth_statistics(ground_truth)
    predictions_sorted_transposed = np.vstack((predictions_one, predictions_two))[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    return calc_pvalue(aucs, delongcov)



In [ ]:
# 计算类别权重
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
# 设置模型参数
params = {
  'alpha': 5.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}
xgb_model = xgb.XGBClassifier(**params)

# 初始化交叉验证器
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 初始化前向选择的变量
y_pred_lst_prev1 = np.zeros(len(df_group))
y_pred_lst_prev2 = np.zeros(len(df_group))
y_pred_lst_prev3 = np.zeros(len(df_group))
tmp_f, AUC_cv_lst = [], []


# 顺序前向选择的过程
for f in top_features:  # 遍历所有筛选出的特征
    tmp_f.append(f)  # 依次添加一个特征
    my_X = df_feature[tmp_f]  # 选择当前的特征集合

    AUC_cv, y_pred_lst, y_true_lst = [], [], []

    # 交叉验证
    for train_idx, test_idx in cv.split(my_X, df_group):
        X_train, X_test = my_X.iloc[train_idx, :], my_X.iloc[test_idx, :]
        y_train, y_test = df_group.iloc[train_idx], df_group.iloc[test_idx]

        # 训练GBDT模型
        my_xgb = xgb.XGBClassifier(**params)
        my_xgb.fit(X_train, y_train)

        # 预测概率
        y_pred_prob = my_xgb.predict_proba(X_test)[:, 1]
        AUC_cv.append(roc_auc_score(y_test, y_pred_prob))  # 计算AUC

        y_pred_lst += y_pred_prob.tolist()
        y_true_lst += y_test.tolist()

    # 计算整体的AUC
    auc_full = roc_auc_score(y_true_lst, y_pred_lst)

    # Delong检验：评估新特征的显著性提升
    log10_p1 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev1), np.array(y_pred_lst))
    log10_p2 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev2), np.array(y_pred_lst))
    log10_p3 = delong_roc_test(np.array(y_true_lst), np.array(y_pred_lst_prev3), np.array(y_pred_lst))

    print(f"Feature: {f}, Delong p-values: {log10_p1}, {log10_p2}, {log10_p3}")

    # 更新前一轮预测结果
    y_pred_lst_prev3 = y_pred_lst_prev2
    y_pred_lst_prev2 = y_pred_lst_prev1
    y_pred_lst_prev1 = y_pred_lst


    tmp_out = np.array([np.mean(AUC_cv), np.std(AUC_cv), 10**log10_p1[0][0], 10**log10_p2[0][0], 10**log10_p3[0][0], auc_full])
    AUC_cv_lst.append(tmp_out)
    print(f"Feature: {f}, AUC Results: {tmp_out}")

In [ ]:
 # 输出当前特征集的AUC结果和显著性检验结果
tmp_out = np.array([
    np.mean(AUC_cv),
    np.std(AUC_cv),
    10**log10_p1[0][0],  # 转换为标量
    10**log10_p2[0][0],  # 转换为标量
    10**log10_p3[0][0],  # 转换为标量
    auc_full
])
print(f"Feature: {f}, AUC Results: {tmp_out}")

# 创建结果DataFrame
AUC_df = pd.DataFrame(AUC_cv_lst, columns=['AUC_mean', 'AUC_std', 'Delong1', 'Delong2', 'Delong3', 'AUC_all'])
AUC_df[['AUC_mean', 'AUC_std', 'AUC_all']] = np.round(AUC_df[['AUC_mean', 'AUC_std', 'AUC_all']], 3)

# 添加特征名称
AUC_df = pd.concat((pd.DataFrame({'Analytes': tmp_f}), AUC_df), axis=1)

# 保存到CSV文件
outfile = dpath + 'Delong_Selection_Results2.csv'
AUC_df.to_csv(outfile, index=False)

print('Finished feature selection and saved results.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(dpath + 'Top_40_InfoGain_features.csv', usecols = ['Analytes', 'Ensemble_cv'])
imp_df.rename(columns = {'Ensemble_cv': 'sRNA_imp'}, inplace = True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how = 'left', on = ['Analytes'])
mydf

In [ ]:
def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while((p_lst[i]<0.05)|(p_lst[i+1]<0.05)):
        i+=1
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper']>=1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf)+1)]
nb_f = get_nb_f(mydf)

fig, ax = plt.subplots(figsize = (18, 6.5))
palette = sns.color_palette("Blues",n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x = "Analytes", y = "sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r']*nb_f + ['k']*(len(mydf)-nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
#ax.set_title(my_title, y=1.0, pad=-25, weight='bold', fontsize=24)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5,  linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f+1), mydf['AUC_mean'][:nb_f+1], 'red', alpha = 0.8, marker='o')
ax2.plot(np.arange(nb_f+1, len(mydf)), mydf['AUC_mean'][nb_f+1:], 'black', alpha = 0.8, marker='o')
ax2.plot([nb_f, nb_f+1], mydf['AUC_mean'][nb_f:nb_f+2], 'black', alpha = 0.8, marker='o')
plt.fill_between(mydf['rna_idx']-1, mydf['AUC_lower'], mydf['AUC_upper'], color = 'tomato', alpha = 0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])


fig.tight_layout()
plt.xlim([-.6, len(mydf)-.2])
plt.savefig(dpath+'Delong_Selection_Plot.svg', dpi=300, format='svg')
plt.show()

In [ ]:
def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while((p_lst[i]<0.05)|(p_lst[i+1]<0.05)):
        i+=1
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper']>=1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf)+1)]
nb_f = get_nb_f(mydf)

fig, ax = plt.subplots(figsize = (18, 6.5))
palette = sns.color_palette("Blues",n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x = "Analytes", y = "sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r']*nb_f + ['k']*(len(mydf)-nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
#ax.set_title(my_title, y=1.0, pad=-25, weight='bold', fontsize=24)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5,  linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f+1), mydf['AUC_mean'][:nb_f+1], 'red', alpha = 0.8, marker='o')
ax2.plot(np.arange(nb_f+1, len(mydf)), mydf['AUC_mean'][nb_f+1:], 'black', alpha = 0.8, marker='o')
ax2.plot([nb_f, nb_f+1], mydf['AUC_mean'][nb_f:nb_f+2], 'black', alpha = 0.8, marker='o')
plt.fill_between(mydf['rna_idx']-1, mydf['AUC_lower'], mydf['AUC_upper'], color = 'tomato', alpha = 0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])


fig.tight_layout()
plt.xlim([-.6, len(mydf)-.2])
plt.savefig(dpath+'Delong_Selection_Plot.svg', dpi=300, format='svg')
plt.show()

In [ ]:
# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

fig, ax = plt.subplots(figsize=(18, 6.5))
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False), palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])
plt.savefig(dpath+'Delong_Selection_Plot_top10.svg', dpi=300, format='svg')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 假设 top_features_df 是您的DataFrame
# 计算每个特征的平均 SHAP 值
top_features_df['abs_shap'] = np.abs(top_features_df['ShapValues_cv'])

# 选择绝对值最大的前20个特征
top20_features = top_features_df.nlargest(20, 'abs_shap')

# 提取前20个特征的名称和对应的 SHAP 值
feature_names = top20_features['Analytes']
shap_values = top20_features['ShapValues_cv']

# 创建 SHAP 贡献图
plt.figure(figsize=(10, 8))
sns.barplot(x='ShapValues_cv', y='Analytes', data=top20_features)
plt.title('Top 20 Features by SHAP Contribution')
plt.xlabel('SHAP Value')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
if 2:
    print(5)
else :
    print(6)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import joblib

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import os
import xgboost as xgb
import joblib
import numpy as np
import pandas as pd
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.model_selection import train_test_split, StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer

In [ ]:
dpath = "D:/UKB/four/"
outfile = os.path.join(dpath, "XGB_feature_importance.csv")
model_path = os.path.join(dpath, "best_xgb_model.pkl")  # 模型保存路径
df = pd.read_csv(r"D:\Rdata and workplace\小人群显著.csv")
df

In [ ]:

england_indices = df[df['Region'] == 'England'].index
X_england = df.loc[england_indices].drop(columns=[ 'status', 'Region','time','Participant.ID'])
y_england = df.loc[england_indices, 'status']  # 
X_train = X_england
y_train = y_england
df_group = y_train
df_feature = X_train

In [ ]:


# 筛选来自苏格兰和威尔士的参与者
external_indices = df[df['Region'].isin(['Scotland', 'Wales'])].index
# 获取苏格兰和威尔士参与者的数据
X_external = df.loc[external_indices].drop(columns=[ 'status', 'Region','time','Participant.ID'])
y_external = df.loc[external_indices, 'status']  # 标签
print("外部验证集形状:", X_external.shape)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from joblib import dump
from sklearn.metrics import roc_auc_score  # 添加这一行

# 初始化参数
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'alpha': 6.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

# 存储数据容器
best_model_info = {
    'model': None,
    'fold': 0,
    'auc': 0,
    'val_idx': None,  # 存储验证集索引
    'fpr': None,
    'tpr': None
}
cv_metrics = []
all_fpr = []
all_tpr = []

# ================= 交叉验证流程 =================
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold + 1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # 模型训练
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    # 计算指标
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)

    # 存储结果
    cv_metrics.append(fold_auc)
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    all_fpr.append(fpr)
    all_tpr.append(tpr)

    # 更新最佳模型
    if fold_auc > best_model_info['auc']:
        best_model_info.update({
            'model': model,
            'fold': fold + 1,
            'auc': fold_auc,
            'val_idx': val_idx,  # 保存验证集索引
            'fpr': fpr,
            'tpr': tpr
        })
        print(f"New best model at Fold {fold + 1}, AUC = {fold_auc:.4f}")

# 保存最佳模型
dump(best_model_info['model'], 'best_model.joblib')
print(f"\nBest model from Fold {best_model_info['fold']}, AUC = {best_model_info['auc']:.4f}")
train_scores = best_model_info['model'].predict_proba(X_train)[:, 1]
external_scores = best_model_info['model'].predict_proba(X_external)[:, 1]

# 创建包含预测结果的数据框
results = pd.DataFrame({
    'Participant.ID': pd.concat([
        df.loc[england_indices, 'Participant.ID'], 
        df.loc[external_indices, 'Participant.ID']
    ]).values,
    'Sex': pd.concat([
        df.loc[england_indices, 'Sex'], 
        df.loc[external_indices, 'Sex']
    ]).values,
    'Age': pd.concat([
        df.loc[england_indices, 'Age.at.recruitment'], 
        df.loc[external_indices, 'Age.at.recruitment']
    ]).values,
    'status': pd.concat([y_train, y_external]).values,
    'AUC_Score': np.concatenate([train_scores, external_scores]),
    'Region': pd.concat([
        pd.Series(['England']*len(england_indices)),
        df.loc[external_indices, 'Region']
    ]).values
})

# 保存完整结果
results.to_csv('25all_participants_auc_scores.csv', index=False)
print("\n已保存所有参与者的AUC预测分数到 25all_participants_auc_scores.csv")
# ================= ROC曲线可视化 =================
plt.figure(figsize=(6, 6))

# 绘制各折ROC曲线
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i in range(5):
    plt.plot(all_fpr[i], all_tpr[i],
             color=colors[i], lw=1, alpha=0.3,
             label=f'Fold {i + 1} (AUC = {cv_metrics[i]:.2f})')

# 计算平均ROC曲线
mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(5):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= 5
mean_auc = auc(mean_fpr, mean_tpr)

# 绘制平均曲线
plt.plot(mean_fpr, mean_tpr, color='b', lw=2,
         label=f'Mean AUC = {mean_auc:.2f} ')

# 绘制外部验证曲线
y_ext_proba = best_model_info['model'].predict_proba(X_external)[:, 1]
fpr_ext, tpr_ext, _ = roc_curve(y_external, y_ext_proba)
roc_auc_ext = auc(fpr_ext, tpr_ext)
plt.plot(fpr_ext, tpr_ext, color='g', lw=2,
         label=f'External (AUC = {roc_auc_ext:.2f})')

# 格式设置
plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves', fontsize=14)
plt.legend(loc='lower right', frameon=True, facecolor='white')
plt.grid(alpha=0.3)
plt.tight_layout()

# 保存为PDF（关键修改：在plt.show()之前保存）
plt.savefig('小人群显著roc_curve.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 混淆矩阵可视化 =================
# 获取最佳折数据
X_val_best = X_train.iloc[best_model_info['val_idx']]
y_val_best = y_train.iloc[best_model_info['val_idx']]

# 生成预测结果
y_val_pred = best_model_info['model'].predict(X_val_best)
y_ext_pred = best_model_info['model'].predict(X_external)

# 创建画布
plt.figure(figsize=(12, 5))

# 内部验证矩阵
plt.subplot(1, 2, 1)
cm_val = confusion_matrix(y_val_best, y_val_pred)
disp_val = ConfusionMatrixDisplay(cm_val, display_labels=['Healthy', 'Stroke'])
disp_val.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title(f'Best Fold ({best_model_info["fold"]})\nAUC = {best_model_info["auc"]:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_val[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_val[i, j] > cm_val.max() / 2 else "black")

# 外部验证矩阵
plt.subplot(1, 2, 2)
cm_ext = confusion_matrix(y_external, y_ext_pred)
disp_ext = ConfusionMatrixDisplay(cm_ext, display_labels=['Healthy', 'Stroke'])
disp_ext.plot(cmap='Oranges', ax=plt.gca(), colorbar=False)
plt.title(f'External Validation\nAUC = {roc_auc_ext:.2f}')

# 添加数值标签
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_ext[i, j]}",
                 ha="center", va="center",
                 color="white" if cm_ext[i, j] > cm_ext.max() / 2 else "black")

plt.tight_layout()

# 保存为PDF（关键修改：在plt.show()之前保存）
plt.savefig('小人群显著confusion_matrix.pdf', format='pdf', bbox_inches='tight', facecolor='white', dpi=300)
plt.show()  # 显示图形（可选）

# ================= 性能报告 =================
print("\n" + "=" * 55)
print(f"{' Internal Validation Report ':=^55}")
print(classification_report(y_val_best, y_val_pred, target_names=['Healthy', 'Stroke']))

print("\n" + "=" * 55)
print(f"{' External Validation Report ':=^55}")
print(classification_report(y_external, y_ext_pred, target_names=['Healthy', 'Stroke']))


In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, classification_report

# 假设您已经有以下数据：
# df - 包含原始数据（含Sex列）
# best_model_info['model'] - 训练好的模型
# england_indices, external_indices - 索引

# 1. 为所有样本生成预测概率
train_scores = best_model_info['model'].predict_proba(X_train)[:, 1]
external_scores = best_model_info['model'].predict_proba(X_external)[:, 1]

# 2. 创建包含预测结果的数据框（按性别分组）
results = pd.DataFrame({
    'Participant.ID': pd.concat([
        df.loc[england_indices, 'Participant.ID'], 
        df.loc[external_indices, 'Participant.ID']
    ]),
    'Sex': pd.concat([
        df.loc[england_indices, 'Sex'],
        df.loc[external_indices, 'Sex']
    ]),
    'time': pd.concat([
        df.loc[england_indices, 'time'],
        df.loc[external_indices, 'time']
    ]),
    'status': pd.concat([y_train, y_external]),
    'AUC_Score': np.concatenate([train_scores, external_scores]),
    'Cluster': pd.concat([
        pd.Series([1]*len(england_indices)),
        pd.Series([2]*len(external_indices))
    ])
})

# 3. 计算整体和按性别的AUC
print("整体AUC:")
print(f"训练集: {roc_auc_score(y_train, train_scores):.4f}")
print(f"验证集: {roc_auc_score(y_external, external_scores):.4f}\n")

print("按性别分组的AUC:")
for sex in results['Sex'].unique():
    subset = results[results['Sex'] == sex]
    auc = roc_auc_score(subset['status'], subset['AUC_Score'])
    print(f"{sex}: {auc:.4f} (n={len(subset)})")

# 4. 按性别的分类报告
threshold = 0.5
results['Predicted_Status'] = (results['AUC_Score'] >= threshold).astype(int)

print("\n按性别分类报告（阈值=0.5）：")
for sex in results['Sex'].unique():
    subset = results[results['Sex'] == sex]
    print(f"\n--- {sex} ---")
    print(classification_report(
        subset['status'], 
        subset['Predicted_Status'],
        target_names=["Alive", "Dead"]
    ))

# 5. 保存结果
results.to_csv("individual_auc_scores_by_sex.csv", index=False)
print("\n已保存按性别的AUC打分到 individual_auc_scores_by_sex.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
dpath = "D:/UKB/four/"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv", usecols=['Analytes', 'Ensemble_cv'])
imp_df.rename(columns={'Ensemble_cv': 'sRNA_imp'}, inplace=True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how='left', on=['Analytes'])
mydf


def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while ((p_lst[i] < 0.05) | (p_lst[i + 1] < 0.05)):
        i += 1
    return i


# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper'] >= 1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf) + 1)]
nb_f = get_nb_f(mydf)

fig, ax = plt.subplots(figsize=(18, 6.5))
palette = sns.color_palette("Blues", n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r'] * nb_f + ['k'] * (len(mydf) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('Feature importance', weight='bold', fontsize=18)
#ax.set_title(my_title, y=1.0, pad=-25, weight='bold', fontsize=24)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf)), mydf['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')
ax2.plot([nb_f, nb_f + 1], mydf['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')
plt.fill_between(mydf['rna_idx'] - 1, mydf['AUC_lower'], mydf['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

fig.tight_layout()
plt.xlim([-.6, len(mydf) - .2])
plt.savefig(dpath + 'Delong_Selection_Plot.svg', dpi=300, format='svg')
plt.show()
# 获取前十个数据
mydf_top10 = mydf[:10]


# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i


nb_f = get_nb_f(mydf_top10)

fig, ax = plt.subplots(figsize=(18, 6.5))
palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
            palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])
plt.savefig(dpath + 'Delong_Selection_Plot_top10.svg', dpi=300, format='svg')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv", usecols=['Analytes', 'Ensemble_cv'])
imp_df.rename(columns={'Ensemble_cv': 'sRNA_imp'}, inplace=True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how='left', on=['Analytes'])

def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while ((p_lst[i] < 0.05) | (p_lst[i + 1] < 0.05)):
        i += 1
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper'] >= 1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf) + 1)]
nb_f = get_nb_f(mydf)

# 创建图形，设置背景为白色，添加黑色边框
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')
ax.set_frame_on(True)  # 添加边框
ax.patch.set_facecolor('white')  # 明确设置背景为白色
ax.patch.set_edgecolor('black')  # 边框颜色为黑色
ax.patch.set_linewidth(1.5)  # 边框线宽

palette = sns.color_palette("Blues", n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r'] * nb_f + ['k'] * (len(mydf) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('Feature importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf)), mydf['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')
ax2.plot([nb_f, nb_f + 1], mydf['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')
plt.fill_between(mydf['rna_idx'] - 1, mydf['AUC_lower'], mydf['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

fig.tight_layout()
plt.xlim([-.6, len(mydf) - .2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='black', dpi=300, transparent=False)
plt.show()

# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

# 创建图形，设置背景为白色，添加黑色边框
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')

ax.patch.set_facecolor('white')  # 明确设置背景为白色

palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
            palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot_top10.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='black', dpi=300, transparent=False)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv", usecols=['Analytes', 'Ensemble_cv'])
imp_df.rename(columns={'Ensemble_cv': 'sRNA_imp'}, inplace=True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how='left', on=['Analytes'])

def get_nb_f(mydf):
    p_lst = mydf.Delong2.tolist()
    i = 0
    while ((p_lst[i] < 0.05) | (p_lst[i + 1] < 0.05)):
        i += 1
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper'] >= 1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf) + 1)]
nb_f = get_nb_f(mydf)

# 创建图形，设置背景为白色
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')
ax.patch.set_facecolor('white')  # 明确设置背景为白色

palette = sns.color_palette("Blues", n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r'] * nb_f + ['k'] * (len(mydf) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('Feature importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf)), mydf['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')
ax2.plot([nb_f, nb_f + 1], mydf['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')
plt.fill_between(mydf['rna_idx'] - 1, mydf['AUC_lower'], mydf['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

fig.tight_layout()
plt.xlim([-.6, len(mydf) - .2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='none', dpi=300, transparent=False)
plt.show()

# 获取前十个数据
mydf_top10 = mydf[:10]

# 修改后的 get_nb_f 函数
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

nb_f = get_nb_f(mydf_top10)

# 创建图形，设置背景为白色
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')

ax.patch.set_facecolor('white')  # 明确设置背景为白色

palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
            palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot_top10.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='none', dpi=300, transparent=False)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# 加载之前保存的AUC结果DataFrame
imp_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv", usecols=['Analytes', 'Ensemble_cv'])
imp_df.rename(columns={'Ensemble_cv':'sRNA_imp'}, inplace=True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how='left', on=['Analytes'])

def get_nb_f(mydf):
    if len(mydf) < 2:
        print("数据不足，无法进行计算")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:  # 确保 i + 1 不会超出范围
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

# 计算AUC的上下限
mydf['AUC_lower'] = mydf['AUC_mean'] - mydf['AUC_std']
mydf['AUC_upper'] = mydf['AUC_mean'] + mydf['AUC_std']
mydf['AUC_upper'].iloc[mydf['AUC_upper'] >= 1] = 1
mydf['rna_idx'] = [i for i in range(1, len(mydf) + 1)]
nb_f = get_nb_f(mydf)

# 创建图形，设置背景为白色
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')
ax.patch.set_facecolor('white')  # 明确设置背景为白色

palette = sns.color_palette("Blues", n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", palette=palette, data=mydf.sort_values(by="sRNA_imp", ascending=False))
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(mydf['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')
my_col = ['r'] * nb_f + ['k'] * (len(mydf) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('Feature importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf)), mydf['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')
ax2.plot([nb_f, nb_f + 1], mydf['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')
plt.fill_between(mydf['rna_idx'] - 1, mydf['AUC_lower'], mydf['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

fig.tight_layout()
plt.xlim([-.6, len(mydf) - .2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='none', dpi=300, transparent=False)
plt.show()

# 获取前十个数据
mydf_top10 = mydf[:10]

# 假设X是特征矩阵，y是目标变量，这里需要根据你的实际数据进行替换
# 例如，如果'mydf_top10'中除了'Analytes'列都是特征，且没有目标变量（SHAP值计算有时不需要目标变量）
X = mydf_top10.drop(columns=['Analytes'])
# 这里使用一个简单的模型示例，你需要根据实际情况选择合适的模型
model = shap.linear_model.LinearModel(X)
# 计算SHAP值
explainer = shap.Explainer(model)
shap_values = explainer(X)

# 按照Delong图的顺序绘制SHAP摘要图
shap.summary_plot(shap_values.values, X, plot_type="bar", 
                    feature_names=X.columns, 
                    color=palette[:len(X.columns)], 
                    show=False)
plt.title("SHAP Summary Plot for Top 10 Features", fontsize=18)
plt.xlabel("SHAP Value (Magnitude)", fontsize=14)
plt.ylabel("Features", fontsize=14)
plt.tight_layout()
shap_save_path = r"C:\Users\lenovo\Desktop\Figure\SHAP_Summary_Plot_top10.pdf"
plt.savefig(shap_save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='none', dpi=300, transparent=False)
plt.show()

nb_f = get_nb_f(mydf_top10)

# 创建图形，设置背景为白色
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')

ax.patch.set_facecolor('white')  # 明确设置背景为白色

palette = sns.color_palette("Blues", n_colors=len(mydf_top10))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", data=mydf_top10.sort_values(by="sRNA_imp", ascending=False),
            palette=palette)
y_imp_up_lim = round(mydf_top10['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.tick_params(axis='y', labelsize=14)

# 设置刻度并旋转标签
ax.set_xticks(range(len(mydf_top10)))
ax.set_xticklabels(mydf_top10['Analytes'], rotation=45, fontsize=10, horizontalalignment='right')

my_col = ['r'] * nb_f + ['k'] * (len(mydf_top10) - nb_f)
for ticklabel, tickcolor in zip(plt.gca().get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

ax.set_ylabel('sRNA importance', weight='bold', fontsize=18)
ax.set_xlabel('')
ax.grid(which='minor', alpha=0.2, linestyle=':')
ax.grid(which='major', alpha=0.5, linestyle='--')
ax.set_axisbelow(True)

# 绘制第二个 y 轴上的 AUC 曲线
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf_top10['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, marker='o')
ax2.plot(np.arange(nb_f + 1, len(mydf_top10)), mydf_top10['AUC_mean'][nb_f + 1:], 'black', alpha=0.8, marker='o')

# 确保索引不会超出范围
if nb_f + 2 > len(mydf_top10):
    ax2.plot([nb_f], mydf_top10['AUC_mean'][nb_f:nb_f + 1], 'black', alpha=0.8, marker='o')
else:
    ax2.plot([nb_f, nb_f + 1], mydf_top10['AUC_mean'][nb_f:nb_f + 2], 'black', alpha=0.8, marker='o')

# 填充AUC区间
plt.fill_between(mydf_top10.index, mydf_top10['AUC_lower'], mydf_top10['AUC_upper'], color='tomato', alpha=0.2)
ax2.set_ylabel('Cumulative AUC', weight='bold', fontsize=18)
ax2.tick_params(axis='y', labelsize=14)
y_auc_up_lim = round(mydf_top10['AUC_upper'].max() + 0.01, 2)
y_auc_low_lim = round(mydf_top10['AUC_lower'].min() - 0.01, 2)
ax2.set_ylim([y_auc_low_lim, y_auc_up_lim])

# 调整布局并保存图像
fig.tight_layout()
plt.xlim([-0.6, len(mydf_top10) - 0.2])

# 保存为PDF - 设置背景为白色，无阴影
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot_top10.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='none', dpi=300, transparent=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from matplotlib import rcParams

# Set global Arial font with SCI standard settings
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 12
rcParams['axes.titlesize'] = 16
rcParams['axes.labelsize'] = 14
rcParams['xtick.labelsize'] = 12
rcParams['ytick.labelsize'] = 12
rcParams['legend.fontsize'] = 12

# Load data
imp_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv", usecols=['Analytes', 'Ensemble_cv'])
imp_df.rename(columns={'Ensemble_cv':'sRNA_imp'}, inplace=True)
AUC_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')
mydf = pd.merge(AUC_df, imp_df, how='left', on=['Analytes'])

# Function to determine significant features
def get_nb_f(mydf):
    if len(mydf) < 2:
        print("Insufficient data for calculation")
        return 0
    p_lst = mydf.Delong2.tolist()
    i = 0
    while i < len(p_lst) - 1:
        if p_lst[i] < 0.05 or p_lst[i + 1] < 0.05:
            i += 1
        else:
            break
    return i

# Create figure with proper borders
fig, ax = plt.subplots(figsize=(18, 6.5), facecolor='white')
ax.set_facecolor('white')

# Add black border
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# Bar plot with improved styling
palette = sns.color_palette("Blues", n_colors=len(mydf))
palette.reverse()
sns.barplot(ax=ax, x="Analytes", y="sRNA_imp", 
            palette=palette, 
            data=mydf.sort_values(by="sRNA_imp", ascending=False),
            edgecolor='black', linewidth=0.5)

# Formatting
y_imp_up_lim = round(mydf['sRNA_imp'].max() + 0.01, 2)
ax.set_ylim([0, y_imp_up_lim])
ax.set_ylabel('Feature Importance', weight='bold')
ax.set_xlabel('')
ax.set_xticklabels(mydf['Analytes'], rotation=45, ha='right')

# Color significant features
nb_f = get_nb_f(mydf)
my_col = ['r'] * nb_f + ['k'] * (len(mydf) - nb_f)
for ticklabel, tickcolor in zip(ax.get_xticklabels(), my_col):
    ticklabel.set_color(tickcolor)

# Grid lines
ax.grid(True, which='both', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)

# AUC plot without confidence intervals
ax2 = ax.twinx()
ax2.plot(np.arange(nb_f + 1), mydf['AUC_mean'][:nb_f + 1], 'red', alpha=0.8, 
         marker='o', markersize=8, linewidth=2)
ax2.plot(np.arange(nb_f + 1, len(mydf)), mydf['AUC_mean'][nb_f + 1:], 'black', 
         alpha=0.8, marker='o', markersize=8, linewidth=2)
ax2.plot([nb_f, nb_f + 1], mydf['AUC_mean'][nb_f:nb_f + 2], 'black', 
         alpha=0.8, marker='o', markersize=8, linewidth=2)

ax2.set_ylabel('Cumulative AUC', weight='bold')
y_auc_up_lim = round(mydf['AUC_mean'].max() + 0.05, 2)
y_auc_low_lim = round(mydf['AUC_mean'].min() - 0.05, 2)
ax2.set_ylim([max(0.5, y_auc_low_lim), min(1.0, y_auc_up_lim)])

# Add border to second axis
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.tight_layout()
plt.xlim([-.6, len(mydf) - .2])

# Save with high quality
save_path = r"C:\Users\lenovo\Desktop\Figure\Delong_Selection_Plot.pdf"
plt.savefig(save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='black', dpi=600)
plt.show()

# Top 10 features SHAP plot
mydf_top10 = mydf[:10]

# Create SHAP summary plot with proper formatting
plt.figure(figsize=(10, 6), facecolor='white')
shap.summary_plot(shap_values.values, X, plot_type="bar", 
                 feature_names=X.columns, 
                 color=palette[:len(X.columns)], 
                 show=False)

# Add borders and formatting
ax = plt.gca()
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

plt.title("SHAP Summary Plot for Top 10 Features", weight='bold')
plt.xlabel("SHAP Value (Magnitude)", weight='bold')
plt.ylabel("Features", weight='bold')
plt.grid(True, alpha=0.2)
plt.tight_layout()

shap_save_path = r"C:\Users\lenovo\Desktop\Figure\SHAP_Summary_Plot_top10.pdf"
plt.savefig(shap_save_path, format='pdf', bbox_inches='tight', 
            facecolor='white', edgecolor='black', dpi=600)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from matplotlib import rcParams

# 1. 加载数据
shap_df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv")  # 替换为您的实际路径
delong_df = pd.read_csv(dpath + 'Delong_Selection_Results2.csv')  # 替换为您的实际路径

# 2. 合并数据并排序（按照Delong图的sRNA_imp降序）
merged_df = pd.merge(delong_df, shap_df, on='Analytes', how='left')
ordered_features = merged_df.sort_values(by='Ensemble_cv_x', ascending=False)['Analytes'].tolist()

# 3. 准备模型和SHAP计算（假设已有训练好的模型）
# model = XGBClassifier()  # 您的已训练模型
# X = df[ordered_features]  # 按顺序排列的特征数据

# 计算SHAP值（示例代码，需替换实际数据）
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# 4. 创建SHAP摘要图（按Delong顺序）
plt.figure(figsize=(14, 10), facecolor='white')
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 12

shap.summary_plot(
    shap_values,
    X,
    feature_names=ordered_features,
    max_display=len(ordered_features),
    plot_type="dot",
    show=False,
    plot_size=(12, 8),
    color_bar_label='Feature value'
)

# 5. 美化图形
ax = plt.gca()
ax.set_facecolor('white')
ax.set_xlabel('SHAP Value (Impact on Prediction)', fontsize=14, weight='bold')
ax.set_ylabel('Features', fontsize=14, weight='bold')

# 调整颜色条
cbar = plt.gcf().axes[-1]
cbar.tick_params(labelsize=12)
cbar.set_ylabel('Feature Value', size=12, weight='bold')

# 6. 保存图像
plt.tight_layout()
plt.savefig(r"C:\Users\lenovo\Desktop\Figure\SHAP_Summary_DelongOrder.pdf", 
            bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

# 7. 创建水平条形图对比三种重要性指标
plt.figure(figsize=(16, 12), facecolor='white')

# 准备数据
plot_df = merged_df.sort_values('Ensemble_cv', ascending=False).head(20)
plot_df = plot_df[['Analytes', 'ShapValues_cv', 'TotalGain_cv', 'Ensemble_cv']]
plot_df = plot_df.sort_values('Ensemble_cv', ascending=True)  # 升序排列使最重要特征在顶部

# 创建子图
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharey=True)

# SHAP值重要性
axes[0].barh(plot_df['Analytes'], plot_df['ShapValues_cv'], 
             color=plt.cm.Blues(np.linspace(0.4, 1, len(plot_df))))
axes[0].set_title('SHAP Value Importance', fontsize=14, pad=10)
axes[0].set_xlabel('Mean |SHAP value|', fontsize=12)

# Total Gain重要性
axes[1].barh(plot_df['Analytes'], plot_df['TotalGain_cv'],
             color=plt.cm.Blues(np.linspace(0.4, 1, len(plot_df))))
axes[1].set_title('Total Gain Importance', fontsize=14, pad=10)
axes[1].set_xlabel('Total Gain', fontsize=12)

# Ensemble重要性
axes[2].barh(plot_df['Analytes'], plot_df['Ensemble_cv'],
             color=plt.cm.Blues(np.linspace(0.4, 1, len(plot_df))))
axes[2].set_title('Ensemble Importance', fontsize=14, pad=10)
axes[2].set_xlabel('Ensemble Score', fontsize=12)

# 美化图形
for ax in axes:
    ax.set_facecolor('white')
    ax.grid(axis='x', alpha=0.3)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(0.8)

plt.tight_layout()
plt.savefig(r"C:\Users\lenovo\Desktop\Figure\Feature_Importance_Comparison.pdf", 
            bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
merged_df 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import shap

# 1. 加载数据
df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv")

# 确保列名与文件中的列名匹配
feature_names = df['Analytes'].tolist()
shap_values = df['ShapValues_cv'].tolist()

# 假设您已经有一个特征数据集X，这里我们使用一个空的DataFrame作为示例
# 在实际应用中，X应该是您用于训练模型的特征数据集
X = pd.DataFrame(columns=Analytes)

# 计算SHAP值（这里我们假设shap_values已经是计算好的SHAP值）
# 由于shap_values是列表，我们将其转换为适合shap.summary_plot的格式
shap_values_array = [shap_values]

# 2. 创建SHAP摘要图
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_array, X, feature_names=feature_names, max_display=len(feature_names), plot_type="bar")

# 美化图形
plt.title("SHAP Summary Plot", fontsize=16)
plt.xlabel("SHAP value (contribution to model output)", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()

# 保存图像
plt.savefig(r"C:\Users\lenovo\Desktop\Figure\SHAP_Summary_Plot.pdf", 
            bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

In [ ]:
import pandas as pd
import shap
import matplotlib.pyplot as plt

# 1. 加载数据
df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv")

# 假设 X_test 是您用于生成 SHAP 值的特征数据集
# 由于您没有提供 X_test，我们将使用 CSV 文件中的特征数据作为示例
# 请替换为实际的特征数据集
X_test = df.drop(columns=['Analytes', 'ShapValues_cv', 'TotalGain_cv', 'Ensemble_cv'])

# SHAP 值
shap_values = df['ShapValues_cv'].values.reshape(-1, -1)  # 确保 SHAP 值是二维数组

# 2. 创建 SHAP 摘要图
plt.figure(figsize=(12, 10))  # 增大画布尺寸
shap.summary_plot(
    shap_values,
    X_test,
    max_display=20,
    plot_type="dot",  # 使用点图更清晰显示 impact 值
    show=False,
    color_bar=True,
    plot_size=(12, 8)  # 调整内部绘图区域大小
)

# 显著调整可视化参数
ax = plt.gca()
ax.set_facecolor('white')

# 1. 去掉网格线
ax.grid(False)
# 获取当前图形对象
fig = plt.gcf()
# 2. 添加坐标轴横线
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', xmin=0, xmax=1)

# 获取颜色条所在的 Axes 对象
cbar_ax = fig.axes[-1]  # 假设颜色条是最后一个 Axes 对象

# 调整颜色条的刻度字体大小
cbar_ax.tick_params(labelsize=14)

# 设置颜色条的标签
cbar_ax.set_ylabel('Feature value', size=18)

# 显示图形
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

# 1. 加载数据
df = pd.read_csv(r"D:\UKB\four\Top_40_InfoGain_features.csv")

# 获取特征名称和 SHAP 值
feature_names = df['Analytes']
shap_values = df['ShapValues_cv'].values

# 创建一个 DataFrame 来模拟特征数据，因为我们只需要特征名称来绘制 SHAP 图
# 在实际应用中，X_test 应该是您用于生成 SHAP 值的特征数据集
X_test = pd.DataFrame(np.random.rand(100, len(feature_names)), columns=feature_names)

# 2. 创建 SHAP 摘要图
plt.figure(figsize=(12, 10))  # 增大画布尺寸
shap.summary_plot(
    shap_values.reshape(1, -1),  # 重塑为二维数组，即使只有一个特征
    X_test,
    max_display=20,
    plot_type="dot",  # 使用点图更清晰显示 impact 值
    show=False,
    color_bar=True,
    plot_size=(12, 8)  # 调整内部绘图区域大小
)
# 显著调整可视化参数
ax = plt.gca()
ax.set_facecolor('white')

# 1. 去掉网格线
ax.grid(False)
# 获取当前图形对象
fig = plt.gcf()
# 2. 添加坐标轴横线
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', xmin=0, xmax=1)

# 获取颜色条所在的 Axes 对象
cbar_ax = fig.axes[-1]  # 假设颜色条是最后一个 Axes 对象

# 调整颜色条的刻度字体大小
cbar_ax.tick_params(labelsize=14)

# 设置颜色条的标签
cbar_ax.set_ylabel('Feature value', size=18)

# 显示图形
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import os
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from joblib import dump
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report

# 1. 加载数据
dpath = "D:/UKB/four/"
outfile = os.path.join(dpath, "XGB_feature_importance.csv")
model_path = os.path.join(dpath, "best_xgb_model.pkl")  # 模型保存路径
df = pd.read_csv(r"D:\Rdata and workplace\小人群.csv")
df

# 2. 划分内部和外部数据集
england_indices = df[df['Region'] == 'England'].index
X_england = df.loc[england_indices].drop(columns=['status', 'Region', 'time', 'Participant.ID'])
y_england = df.loc[england_indices, 'status']  # 
X_train = X_england
y_train = y_england

# 筛选来自苏格兰和威尔士的参与者
external_indices = df[df['Region'].isin(['Scotland', 'Wales'])].index
X_external = df.loc[external_indices].drop(columns=['status', 'Region', 'time', 'Participant.ID'])
y_external = df.loc[external_indices, 'status']  # 标签
print("外部验证集形状:", X_external.shape)

# 初始化参数
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = {
    'alpha': 6.0,  # L1 正则化
    'lambda': 1.5,  # L2 正则化
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 5200,  # 设置为较大的值以便早停能起作用
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
    'n_jobs': 4
}

# 存储数据容器
best_model_info = {
    'model': None,
    'fold': 0,
    'auc': 0,
    'val_idx': None,  # 存储验证集索引
    'fpr': None,
    'tpr': None
}
cv_metrics = []
all_fpr = []
all_tpr = []

# ================= 交叉验证流程 ================= 
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n======= Fold {fold + 1} =======")
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # 模型训练
    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    # 计算指标
    y_proba = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, y_proba)

    # 存储结果
    cv_metrics.append(fold_auc)
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    all_fpr.append(fpr)
    all_tpr.append(tpr)

    # 更新最佳模型
    if fold_auc > best_model_info['auc']:
        best_model_info.update({
            'model': model,
            'fold': fold + 1,
            'auc': fold_auc,
            'val_idx': val_idx,  # 保存验证集索引
            'fpr': fpr,
            'tpr': tpr
        })
        print(f"New best model at Fold {fold + 1}, AUC = {fold_auc:.4f}")

# 保存最佳模型
dump(best_model_info['model'], 'best_model.joblib')
print(f"\nBest model from Fold {best_model_info['fold']}, AUC = {best_model_info['auc']:.4f}")
train_scores = best_model_info['model'].predict_proba(X_train)[:, 1]
external_scores = best_model_info['model'].predict_proba(X_external)[:, 1]

# 创建包含预测结果的数据框
results = pd.DataFrame({
    'Participant.ID': pd.concat([
        df.loc[england_indices, 'Participant.ID'], 
        df.loc[external_indices, 'Participant.ID']
    ]).values,
    'Sex': pd.concat([
        df.loc[england_indices, 'Sex'], 
        df.loc[external_indices, 'Sex']
    ]).values,
    'Age': pd.concat([
        df.loc[england_indices, 'Age'], 
        df.loc[external_indices, 'Age']
    ]).values,
    'status': pd.concat([y_train, y_external]).values,
    'AUC_Score': np.concatenate([train_scores, external_scores]),
    'Region': pd.concat([
        pd.Series(['England']*len(england_indices)),
        df.loc[external_indices, 'Region']
    ]).values
})

# 保存完整结果
results.to_csv(outfile, index=False)
print("\n已保存所有参与者的AUC预测分数到", outfile)

# ================= ROC曲线可视化 ================
plt.figure(figsize=(6, 6))

# 绘制各折ROC曲线
colors = ['#1f7b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for i in range(5):
    plt.plot(all_fpr[i], all_tpr[i],
             color=colors[i], lw=1, alpha=0.3,
             label=f'Fold {i + 1} (AUC = {cv_metrics[i]:.2f})')

# 计算平均ROC曲线
mean_fpr = np.linspace(0,1, 100)
mean_tpr = np.zeros_like(mean_fpr)
for i in range(5):
    mean_tpr += np.interp(mean_fpr, all_fpr[i], all_tpr[i])
mean_tpr /= 5
mean_auc = auc(mean_fpr, mean_tpr)




# ================= 性能报告 ================
print("\n" + "=" * 55)
print(f"{' Internal Validation Report ':=^55}")
print(classification_report(y_val_best, y_val_pred, target_names=['Healthy', 'Stroke']))

print("\n" + "=" * 55)
print(f"{' External Validation Report ':=^5}")
print(classification_report(y_external, y_ext_pred, target_names=['Healthy', 'Stroke']))

# 计算 SHAP 值
explainer = shap.TreeExplainer(best_model_info['model'])
shap_values = explainer.shap_values(X_train)

# 创建 SHAP 摘要图
plt.figure(figsize=(12, 10))  # 增大画布尺寸
shap.summary_plot(
    shap_values,  # 直接使用从 CSV 文件加载的 SHAP 值
    X_train,
    max_display=20,
    plot_type="dot",  # 使用点图更清晰显示 impact 值
    show=False,
    color_bar=True,
    plot_size=(12, 8)  # 调整内部绘图区域大小
)

# 显著调整可视化参数
ax = plt.gca()
ax.set_facecolor('white')

# 1. 去掉网格线
ax.grid(False)
# 获取当前图形对象
fig = plt.gcf()
# 2. 添加坐标轴横线
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', xmin=0, xmax=1)

# 获取颜色条所在的 Axes 对象
cbar_ax = fig.axes[-1]  # 假设颜色条是最后一个 Axes 对象

# 调整颜色条的刻度字体大小
cbar_ax.tick_params(labelsize=14)

# 设置颜色条的标签
cbar_ax.set_ylabel('Feature value', size=18)

# 显示图形
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# 1. 初始化SHAP解释器
explainer = shap.TreeExplainer(best_model_info['model'])

# 2. 计算SHAP值（注意：X_train 是 DataFrame）
shap_values = explainer(X_train)

# 3. 绘制SHAP摘要点图（每个样本一个点）
plt.figure(figsize=(10, 8))

# 使用自动调整的max_display显示所有特征，或手动设置（例如：max_display=20）
shap.summary_plot(
    shap_values.values,          # SHAP值矩阵
    X_train,                     # 特征数据
    plot_type="dot",             # 点图模式
    show=False,                  # 不立即显示（以便保存）
    max_display=25,              # 显示最重要的20个特征（可调整）
    color=plt.get_cmap("coolwarm"),  # 颜色映射（红-蓝）
    alpha=0.7,                   # 点透明度
    plot_size=(15, 9)            # 图像大小
)
# 显著调整可视化参数
ax = plt.gca()
ax.set_facecolor('white')
# 4. 美化图表
plt.title("SHAP Summary Plot (Impact on Stroke Prediction)", fontsize=14, pad=20)
plt.xlabel("SHAP Value (Impact on Model Output)", fontsize=12)
plt.gcf().set_facecolor('white')  # 设置背景为白色

# 5. 保存为PDF（矢量图，适合论文）
plt.savefig(
    "小人群显著_SHAP_点图.pdf",
    format="pdf",
    bbox_inches="tight",
    dpi=300,
    facecolor='white'
)

plt.show()

In [ ]:
merged_df.iloc[:, 0].tolist() 

In [ ]:
merged_df.iloc[:, 1].tolist()  # 第一列内容

In [ ]:
import shap
import matplotlib.pyplot as plt

# 1. 初始化解释器并计算SHAP值
explainer = shap.TreeExplainer(best_model_info['model'])
shap_values = explainer(X_train)

custom_order = [
    'GDF15',
    'Sex',
    'Long standing illness disability or infirmity',
    'Diabetes diagnosed by doctor ',
    'TNFRSF10B',
    'Red blood cell erythrocyte distribution width',
    'C reactive',
    'Number of vehicles in household',
    'Own or rent accommodation lived in',
    'L_LDL_CE_pct',
    'Age',
    'Mean corpuscular volume',
    'Glucose',
    'Glycated haemoglobin HbA1c',
    'IDL_CE_pct',
    'Average total household income before tax',
    'Creatinine',
    'Mean reticulocyte volume',
    'Lymphocyte percentage',
    'Testosterone',
    'S_VLDL_FC_by_CE',
    'Usual walking pace',
    'Cystatin C',
    'GlycA',
    'XS_VLDL_PL_pct'
]

# 3. 过滤掉X_train中不存在的特征
available_features = [f for f in custom_order if f in X_train.columns]

# 4. 提取排序后的SHAP值
shap_values_ordered = shap_values[:, available_features]

# 5. 使用beeswarm绘图（自动支持排序）
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(
    shap_values_ordered,
    show=False,
    max_display=len(available_features),
    plot_size=None,
    color=plt.get_cmap("coolwarm")
)


# 6. 调整图表样式
ax = plt.gca()
ax.set_facecolor('white')
plt.title("SHAP Summary (Custom Order)", fontsize=14, pad=20)
plt.xlabel("SHAP Value (Impact on Model Output)", fontsize=12)
plt.gcf().set_facecolor('white')

plt.savefig("SHAP_custom_order.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
from joblib import load
import shap
model = load('best_model.joblib')
model 

In [ ]:
# 获取 Top10 特征名
top10_features = mydf_top10['Analytes'].tolist()  

# 提取训练集和外部验证集的 Top10 特征
X_train_top10 = X_train[top10_features]
X_external_top10 = X_external[top10_features]  # 可选：外部验证数据
top10_features

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np

# Your feature mapping dictionary
feature_mapping = {
    'Age': 'Age at recruitment',
    'ACTA2': 'ACTA2.Actin..aortic.smooth.muscle',
    'WFDC2': 'WFDC2.WAP.four.disulfide.core.domain.protein.2',
    'HAVCR1': 'HAVCR1.Hepatitis.A.virus.cellular.receptor.1',
    'LTBP2': 'LTBP2.Latent.transforming.growth.factor.beta.binding.protein.2',
    'Father still alive': 'Father still alive | Instance 0',
    'XS_VLDL_CE_pct_C': 'XS_VLDL_CE_pct_C',
    'ACE2': 'ACE2.Angiotensin.converting.enzyme.2',
    'MMP12': 'MMP12.Macrophage.metalloelastase',
    'BCAN': 'BCAN.Brevican.core.protein'
}

# Create reverse mapping
reverse_mapping = {v: k for k, v in feature_mapping.items()}

# 1. Get features to display (using full names)
selected_features = list(feature_mapping.values())

# 2. Ensure these features exist in X_test
available_features = [f for f in selected_features if f in X_test.columns]
print("Features to display:", available_features)

# 3. Get abbreviated names
abbreviated_names = [reverse_mapping[f] for f in available_features]

# 4. Get column indices in X_test
feature_indices = [X_test.columns.get_loc(f) for f in available_features]

# 5. Prepare SHAP values
if hasattr(shap_values, 'values'):
    # If it's an Explanation object
    shap_values_selected = shap.Explanation(
        values=shap_values.values[:, feature_indices],
        base_values=shap_values.base_values,
        data=X_test[available_features].values,
        feature_names=abbreviated_names
    )
else:
    # If it's a numpy array, we need to create an Explanation object
    if len(shap_values.shape) == 2:  # For single output
        shap_values_selected = shap.Explanation(
            values=shap_values[:, feature_indices],
            base_values=np.array([shap_values.base_value]) if hasattr(shap_values, 'base_value') else None,
            data=X_test[available_features].values,
            feature_names=abbreviated_names
        )
    else:  # For multi-output
        # Assuming 